# Hands-on: LLM-as-a-Judge für Softwarequalität

Habt ihr schon mal selber einen Judge-Prompt, bzw. eine Review-Anweisung an ein LLM verfasst? Was sollte in solchen Prompts allgemein alles drinnestehen, um möglichst alle Fehler im Code zu finden? Verfasst folgend einen möglichen Prompt für einen LLM-Judge.

Führt dafür die untenstehende Zeile aus.

In [4]:
from pathlib import Path

import ipywidgets as widgets
from IPython.display import HTML, display

if Path("../pyproject.toml").is_file():
    FIRST_PROMPT_PATH = Path("../prompts/first_prompt.md").resolve()
else:
    REPO_DIR = Path("/content/software-quality-engineering")

    if not REPO_DIR.is_dir():
        import subprocess
        subprocess.run(["git", "clone", "-q", "--depth", "1",
                        "https://github.com/evabowe/software-quality-engineering", str(REPO_DIR)], check=True)
    FIRST_PROMPT_PATH = REPO_DIR / "prompts" / "first_prompt.md"

prompt_box = widgets.Textarea(
    value=FIRST_PROMPT_PATH.read_text(encoding="utf-8") if FIRST_PROMPT_PATH.is_file() else "",
    placeholder="Schreibe einen Judge Prompt...",
    layout=widgets.Layout(width="100%", height="250px"),
)
prompt_box.add_class("first-prompt")
save_button = widgets.Button(description="Speichern", icon="save", button_style="primary")
status = widgets.HTML()


def save_first_prompt(_):
    FIRST_PROMPT_PATH.parent.mkdir(parents=True, exist_ok=True)
    FIRST_PROMPT_PATH.write_text(prompt_box.value, encoding="utf-8")
    status.value = (
        f"✅ Gespeichert unter <code>{FIRST_PROMPT_PATH}</code><br>"
        "Zum Einsehen die Dateien aktualisieren: z.B. in Colab links über 📁 Dateien → Aktualisieren-Symbol."
    )


save_button.on_click(save_first_prompt)
display(prompt_box, widgets.HBox([save_button, status]))

display(HTML("""<script>
new MutationObserver(() => document.querySelectorAll(".first-prompt textarea")
    .forEach(t => t.spellcheck = false)).observe(document.body, {childList: true, subtree: true});
</script>"""))

Textarea(value='', layout=Layout(height='250px', width='100%'), placeholder='Schreibe einen Judge Prompt...', …

# Hands-on: LLM-as-a-Judge für Softwarequalität

Im Paper (Chou 2026) bewerten LLMs Code – aber niemand prüft, **ob sie recht haben**. Wir können das: Unsere sechs Code-Beispiele enthalten **bekannte Defekte**. Damit messen wir, wie viele Defekte ein Judge findet (Recall) und wie oft er Fehlalarm schlägt.

**Ablauf**

| Zeit | Schritt |
|---|---|
| 0–5 min | **1 · Setup**: Zelle ausführen, Gruppe wählen, API-Key eintragen (Gruppe 4 startet zusätzlich sofort das lokale Modell) |
| 5–7 min | **2 · Prognose**: Was erwartet ihr? |
| 7–17 min | **3 · Auftrag 1**: Ein Klick – euer Judge bewertet alle Beispiele in drei Haltungen, mit und ohne Tests |
| 17–32 min | **4 · Auftrag 2**: Prompt-Challenge – schreibt einen besseren Judge-Prompt |
| 32–45 min | Auswertung am Beamer und Abschluss |

Das **Scoreboard** läuft am Beamer; den Link findet ihr auf der Folie.

Zellen startet ihr mit ▶ links neben dem Titel. Den Code darin müsst ihr nicht lesen.

*Hinweis zum Datenschutz:* An die KI-Anbieter geht nur der Beispielcode und euer Prompt. Gespeichert wird nur euer Gruppenname, keine personenbezogenen Daten.

In [ ]:
#@title ⚙️ 1 · Setup (einmal ausführen)

import os
import subprocess
import sys

REPO_URL = "https://github.com/evabowe/software-quality-engineering"
# Lokal (Jupyter im Repo) wird der vorhandene Checkout genutzt, in Colab ein frischer Klon.
if os.path.isfile("../pyproject.toml"):
    REPO_DIR = os.path.abspath("..")
else:
    REPO_DIR = "/content/software-quality-engineering"
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR], check=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)  # die Installation wird sonst erst nach einem Neustart sichtbar

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

from judge_lab import ui

ui.setup_panel()

In [ ]:
#@title 🖥️ Nur Gruppe 4: lokales Modell starten (dauert einige Minuten – gleich zu Beginn starten!)
from judge_lab import ollama_colab

print(ollama_colab.start_local_model())

## Kurz erklärt: Was macht der Judge?

Jeder Judge bekommt einen Prompt aus vier Bausteinen:

| Baustein | Inhalt | Wer bestimmt ihn? |
|---|---|---|
| **[1] Haltung** | Rolle und Strenge: *strict*, *objective* oder *lenient* (aus dem Paper übernommen) | Auftrag 1: fest · Auftrag 2: **ihr** |
| **[2] Bewertungsskala** | 4 Dimensionen nach ISO/IEC 25010 (Korrektheit, Zuverlässigkeit, Wartbarkeit, Lesbarkeit), je 0–10 | fest |
| **[3] Aufgabe** | der Code mit Zeilennummern, optional die Tests | fest |
| **[4] Ausgabeformat** | JSON mit Befunden und Scores, damit wir automatisch auswerten können | fest |

Wir gleichen die gemeldeten Befunde mit den bekannten Defekten ab: Ein Befund **trifft**, wenn er auf die richtige Stelle zeigt. Ein Fehlalarm ist ein Bug-Befund ohne echten Defekt dahinter.

In [ ]:
#@title 🔮 2 · Eure Prognose
ui.prognosis_panel()

In [ ]:
#@title 🔍 Code ansehen (optional)
ui.snippet_viewer()

In [ ]:
#@title 🧪 3 · Auftrag 1: Replikation
ui.assignment1_panel()

### Leitfragen zu Auftrag 1 (im Paar besprechen)

1. Wie unterscheiden sich die **Scores** zwischen *strict*, *objective* und *lenient*?
2. Ändert sich mit der Haltung auch, **wie viele Defekte gefunden** werden – oder nur die Note?
3. Hilft es dem Judge, wenn er die **Tests** sieht? Bei welchen Beispielen besonders?

In [ ]:
#@title 🏆 4 · Auftrag 2: Prompt-Challenge
ui.assignment2_panel()

### Regeln und Tipps für Auftrag 2

**Regeln:** Ziel sind möglichst viele echte Defekte bei möglichst wenigen Fehlalarmen (2 Punkte pro Treffer, −1 pro Fehlalarm). Probeläufe auf einem Beispiel sind unbegrenzt, **offizielle Einreichungen gibt es drei**; sie laufen immer über alle sechs Beispiele. Der Prompt muss allgemein funktionieren – keine Hinweise auf bestimmte Zeilen oder Funktionen. Ihr ersetzt nur Baustein [1]; Skala, Aufgabe und Ausgabeformat werden automatisch angehängt.

**Tipps**

1. **Rolle:** Gebt dem Judge eine Rolle, die *gründlich* statt *streng* ist.
2. **Prüfschritte:** Lasst ihn systematisch vorgehen, z. B. jede Bedingung, jede Schleife und jeden Rückgabewert prüfen.
3. **Randfälle:** Leere Eingaben, Duplikate, Grenzwerte, Terminierung.
4. **Spezifikation zuerst:** Erst lesen, was der Code laut Docstring tun soll, dann den Code dagegen prüfen.
5. **Nur sichere Befunde:** Fehlalarme kosten Punkte – lieber weniger, dafür begründete Befunde.

## Abschluss

Danke fürs Mitmachen! Bitte beantwortet zum Schluss noch einmal die Umfrage vom Anfang (Link auf der Folie):
*„Würdet ihr einem LLM das Code-Review überlassen?“*